1. Based on the frequency of the most rated items computed in Week 6, implement the TopPop recommender system, which always recommends
the same top-k items sorted decreasingly by the number of “high” ratings (e.g., ≥ 3) in the training split, train.tsv.

In [22]:
import pandas as pd

class TopPop:
    def __init__(self, k=10):
        self.k = k
        self.top_items = None

    def fit(self, file_path):
        # Read the training data (no header assumed)
        df = pd.read_csv(file_path, sep='\t')
        # Keep only the high ratings (e.g., >= 3)
        df_high = df[df['rating'] >= 3]
        # Compute the frequency of high ratings per item and sort in decreasing order
        counts = df_high['item_id'].value_counts()
        self.top_items = list(counts.index)

    def recommend(self, user_id):
        # Return the top-k items (this model is user-agnostic)
        return self.top_items[:self.k]

2. Choose at least one neighborhood-based model and one latent factor model that uses the observed user-item ratings in the training set to predict the unobserved ratings. Report your choice of models.

In [23]:
from surprise import Dataset, Reader, KNNWithMeans, SVD
import pandas as pd

# For the neighborhood-based model, I chose the KNNWithMeans collaborative filtering algorithm.
# In particular, the KNNWithMeans implementation from the surprise library is a simple yet effective approach 
# that computes similarities between users (or items) and makes predictions based on the ratings from 
# the most similar neighbors.
#
# For the latent factor model, I selected the Singular Value Decomposition (SVD) algorithm.
# SVD decomposes the user-item rating matrix into latent factors, capturing the underlying associations 
# between users and items. This model has been widely used in collaborative filtering tasks with excellent performance.
#
# Below is an example to illustrate how you might set up and train these models
# using the surprise package:

# Load your training data from the file (assuming train.tsv is in 'Data/clean_train.tsv')
df = pd.read_csv('Data/clean_train.tsv', sep='\t')
reader = Reader(rating_scale=(1, 5))

# Prepare the dataset using the surprise framework
data = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)
trainset = data.build_full_trainset()

3. Use 5-fold cross-validation on the training set to tune the hyperparameters of the chosen models (similarity measure and number of neighbors for the neighborhood-based model; number of latent factors and number of epochs for the latent factor model).

In [24]:
from surprise.model_selection import GridSearchCV

# Define parameter grid for KNNWithMeans (neighborhood-based model)
param_grid_knn = {
    'k': [5, 10, 20],
    'verbose': [False],
    'sim_options': {
        'name': ['cosine', 'msd'],
        'user_based': [True]  # We use user-based collaborative filtering
    }
}

# Define parameter grid for SVD (latent factor model)
param_grid_svd = {
    'n_factors': [20, 50, 100],
    'n_epochs': [20, 50, 100]
}

# Perform 5-fold cross-validation for KNNWithMeans
grid_knn = GridSearchCV(KNNWithMeans, param_grid_knn, measures=['rmse', 'mae'], cv=5)
grid_knn.fit(data)

print("Best KNN parameters (based on RMSE):", grid_knn.best_params['rmse'])
print("Best RMSE for KNN:", grid_knn.best_score['rmse'])

# Perform 5-fold cross-validation for SVD
grid_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=5)
grid_svd.fit(data)

Best KNN parameters (based on RMSE): {'k': 20, 'verbose': False, 'sim_options': {'name': 'cosine', 'user_based': True}}
Best RMSE for KNN: 0.9043990207063264


4. Choose an evaluation measure that is suitable for this task and justify your motivation in using it. Report the optimal hyperparameters together with the scores of your chosen measure, averaged over the 5 folds.

In [25]:
# We choose RMSE (Root Mean Squared Error) as the evaluation measure.
# RMSE penalizes larger errors more than smaller ones, making it a good measure to
# capture the overall predictive accuracy of our recommendation models.
# Lower RMSE indicates that the model predictions are closer to the actual ratings.

print("\nKNNWithMeans Best Hyperparameters (based on RMSE):")
print(grid_knn.best_params['rmse'])
print("Average RMSE for KNNWithMeans:", grid_knn.best_score['rmse'])

print("\nSVD Best Hyperparameters (based on RMSE):")
print(grid_svd.best_params['rmse'])
print("Average RMSE for SVD:", grid_svd.best_score['rmse'])


KNNWithMeans Best Hyperparameters (based on RMSE):
{'k': 20, 'verbose': False, 'sim_options': {'name': 'cosine', 'user_based': True}}
Average RMSE for KNNWithMeans: 0.9043990207063264

SVD Best Hyperparameters (based on RMSE):
{'n_factors': 20, 'n_epochs': 20}
Average RMSE for SVD: 0.8430577972337767


5. Run the models with the optimal hyperparameters to the whole training set.

In [30]:
# Re-train the models on the full training set using the optimal hyperparameters from the grid searches

# For KNNWithMeans
knn_best = grid_knn.best_estimator['rmse']
knn_best.fit(trainset)

# For SVD
svd_best = grid_svd.best_estimator['rmse']
svd_best.fit(trainset)

print("KNNWithMeans model trained with optimal hyperparameters.")
print("SVD model trained with optimal hyperparameters.")

KNNWithMeans model trained with optimal hyperparameters.
SVD model trained with optimal hyperparameters.


6. Use the final models to rank the non-rated items for each user. This ranking will be used for the evaluation part next week.

In [31]:
# Get the set of all item inner ids in the training set
all_items = set(trainset.all_items())

# Dictionaries to store recommendations from each model
recommendations_knn = {}
recommendations_svd = {}

# For each user in the training set:
for u in trainset.all_users():
    # Get inner ids of items the user has already rated
    rated_items = set(item for item, _ in trainset.ur[u])
    # Compute the candidate non-rated items
    non_rated = all_items - rated_items
    # Convert user inner id to raw id
    user_raw = trainset.to_raw_uid(u)
    
    # Lists to store (item, predicted rating)
    preds_knn = []
    preds_svd = []
    
    for i in non_rated:
        # Convert item inner id to raw id
        item_raw = trainset.to_raw_iid(i)
        # Predict ratings using each model
        pred_knn = knn_best.predict(user_raw, item_raw).est
        pred_svd = svd_best.predict(user_raw, item_raw).est
        
        preds_knn.append((item_raw, pred_knn))
        preds_svd.append((item_raw, pred_svd))
    
    # Sort candidate items by predicted rating (in descending order)
    ranked_knn = [item for item, score in sorted(preds_knn, key=lambda x: x[1], reverse=True)]
    ranked_svd = [item for item, score in sorted(preds_svd, key=lambda x: x[1], reverse=True)]
    
    recommendations_knn[user_raw] = ranked_knn
    recommendations_svd[user_raw] = ranked_svd

# Display sample recommendations for a few users
print("Sample recommendations using KNNWithMeans:")
for idx, (user_id, recs) in enumerate(recommendations_knn.items()):
    print(f"User {user_id}: {recs[:10]}")  # Show top 10 recommendations
    if idx == 4:
        break

print("\nSample recommendations using SVD:")
for idx, (user_id, recs) in enumerate(recommendations_svd.items()):
    print(f"User {user_id}: {recs[:10]}")  # Show top 10 recommendations
    if idx == 4:
        break

Sample recommendations using KNNWithMeans:
User AGTVZ7ZSDMTEDLMJGZRC7RFJNDWQ: ['B0C67HCGBR', 'B0B8M5FJB6', 'B00H35YIJE', 'B079TLFL33', 'B000T9L7W2', 'B007MY5BDI', 'B00H4PEMM6', 'B09YDBKT7M', 'B08R5GM6YB', 'B0002D0Q2W']
User AG44UYFDZHI6FRBQSLDWOGIEOY4A: ['B0C67HCGBR', 'B0B8M5FJB6', 'B000T9L7W2', 'B007MY5BDI', 'B08R5GM6YB', 'B079P9LDHN', 'B0BPKTYTPQ', 'B001W99HE8', 'B000J5UEGQ', 'B01DBS2U9G']
User AE7P3HIBI3UDLCJLUPUZQWYVPEWA: ['B005M0MUQK', 'B0BRS6V8G4', 'B078L11275', 'B09R6KV6QX', 'B07YK57N2M', 'B00RX5HQS4', 'B0BQ4HSKC9', 'B086QM1F75', 'B07R3S93K8', 'B09WF82F1V']
User AE5M7M2VYMM3WHJIBLCHHEU7UOXQ: ['B06XB3FQKB', 'B0BPJ4Q6FJ', 'B0B8M5FJB6', 'B079TLFL33', 'B007MY5BDI', 'B00H4PEMM6', 'B09YDBKT7M', 'B0BXT384GR', 'B079P9LDHN', 'B0BPKH4HB2']
User AESDNHQRRZEWTZF7A7TFFFTSNY5Q: ['B0BGQZNQ53', 'B07L5B64RG', 'B07Y94MSG9', 'B0CB98SMQR', 'B079NS31NK', 'B00CIHB8FY', 'B000SHQ1QC', 'B0BQ4HSKC9', 'B00IZA1GI2', 'B000U0DU34']

Sample recommendations using SVD:
User AGTVZ7ZSDMTEDLMJGZRC7RFJNDWQ: ['B0BPJ